# 검수 파이프라인 end-to-end — SAM3 탐지 + bbox 미표시 배치 VLM 설명

### 이 노트북이 다루는 것
크롭 방식과의 비교는 끝났다(`vlm_description_check.ipynb` §4/§4-1 참고, 결론: 배치 방식이 문체·구체성 면에서 원하는 느낌에 더 가까웠다). 이 노트북은 그 비교를 다시 하지 않고, **실제로 쓸 파이프라인 하나만** 끝까지 다룬다:

```
원본 이미지
  -> (1) SAM3로 하자 탐지 (garment 1회 세그먼트 -> 하자 탐지, bbox/유형)
  -> (2) 원본 이미지(bbox 미표시) + 번호별 위치·유형 텍스트로 VLM 배치 설명 1회 호출
  -> (3) 번호별로 파싱 + bbox/유형과 매칭 -> 최종 JSON
```

**출력 JSON에 들어가는 것**: `bbox`(좌표) · `defect_type`/`label`(유형) · `description`(VLM 설명). **`상태`(사용자 응답)는 여기서 만들지 않는다** — AI는 판정만 하고, 사용자가 이 finding을 인정/거부했는지는 BE가 `key`로 관리한다(빨강/초록 박스 전환은 FE가 좌표 기반으로 직접 그리는 것이라 서버가 그린 정적 이미지로는 할 수 없다 — 그래서 좌표를 JSON으로 준다).

### 실행 순서
0. 의존성 설치 -> HF 로그인 -> GPU 확인 -> 설정 -> SAM3/VLM 모델 로드 -> 테스트 이미지
1. Step 1 — SAM3 실시간 탐지(garment 1회 세그먼트 -> 하자 탐지 -> 공통 후처리)
2. Step 2 — bbox 미표시 배치 VLM 설명 1회 호출 + 파싱
3. Step 3 — 최종 JSON 조립(bbox+유형+설명)
4. 이미지 전체 배치 실행 + 결과 저장

**프로덕션(`app/`, `worker/`)은 건드리지 않는다.** 여기서 검증되면 별도 계획 문서로 반영 여부를 결정한다.


## 0. 의존성 설치


In [ ]:
%pip install -q -r "../sam3-condition-check/requirements-sam3.txt"
%pip install -q -r "requirements-vlm.txt"


## 0-1. Hugging Face 로그인 (SAM3 gated repo)


In [ ]:
import os

from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("HF_TOKEN 환경변수로 로그인했습니다.")
else:
    print("HF_TOKEN이 없습니다 — huggingface-cli login을 미리 실행했다면 그대로 진행됩니다.")


## 0-2. CUDA / GPU 확인


In [ ]:
import torch

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))


## 0-3. 경로·설정


In [ ]:
import sys

# 공유 유틸(qc_shared.py)은 이 폴더의 부모(validation/QUAL/)에 있다.
sys.path.insert(0, "..")
# SAM3 호출/크롭 유틸과 검증된 bbox 파이프라인(garment 탐지 -> 하자 탐지 -> 후처리)은
# 새로 만들지 않고 형제 스파이크 폴더의 구현을 그대로 재사용한다.
sys.path.insert(0, "../sam3-parts-segment-check")
sys.path.insert(0, "../sam3-border-overlay-check")

import json as jsonlib
import time

import matplotlib.pyplot as plt
from PIL import Image

import qc_shared
from border_overlay_check import run_one
from sam3_caller import Sam3Caller

import model_registry
from loaders import (
    describe_defects_batch,
    load_qwen_vl,
    missing_batch_numbers,
    parse_batch_response,
)

qc_shared.use_korean_font()  # 그래프 제목 한글이 두부(□□)로 깨지지 않게


## 0-4. SAM3 모델 로드 (하자 탐지용)


In [ ]:
sam3_caller = Sam3Caller()
sam3_caller._ensure_loaded()


## 0-5. VLM 모델 로드 (설명 생성용)


In [ ]:
print(f"모델: {model_registry.ACTIVE_VLM} ({model_registry.VLM_MODELS[model_registry.ACTIVE_VLM]})")
vlm_bundle = load_qwen_vl(
    model_registry.VLM_MODELS[model_registry.ACTIVE_VLM],
    quantization=model_registry.ACTIVE_QUANTIZATION,
)
print("VLM 로드 완료")


## 0-6. 테스트 이미지 로드


In [ ]:
image_paths = qc_shared.list_images()
assert image_paths, "sample_images/ 에 이미지가 없습니다 (qc_shared.SHARED_IMAGE_DIR 참고)."
print(f"이미지 {len(image_paths)}장: {[p.name for p in image_paths]}")

# §1~§3은 이 기준 이미지 1장으로 단계별로 확인한다. §4에서 전체 배치를 돈다.
demo_path = image_paths[0]
demo_image = Image.open(demo_path).convert("RGB")

plt.figure(figsize=(5, 5))
plt.imshow(demo_image)
plt.title(f"기준 이미지: {demo_path.name}")
plt.axis("off")
plt.show()


## 1. Step 1 — SAM3 실시간 탐지

`sam3-border-overlay-check`의 검증된 파이프라인(옷 1회 세그먼트 -> 하자 탐지 -> 공통 후처리, bbox 유지로 결정된 그 방식)을 그대로 재사용한다 — 여기서 새로 탐지 로직을 만들지 않는다.


In [ ]:
result = run_one(sam3_caller, demo_image)
kept = result["kept"]
print(f"하자 {len(kept)}건 탐지, garment_found={result['garment_found']}")
for inst in kept:
    label = qc_shared.KOREAN_LABELS.get(inst["type"], inst["type"])
    print(f"  [{inst['no']}] {label}  bbox={[round(v) for v in inst['bbox']]}")


## 2. Step 2 — bbox 미표시 배치 VLM 설명 1회 호출

**원본 이미지 그대로**(bbox 안 그림) + 번호별 위치·유형을 알려주는 텍스트 프롬프트로 한 번만 호출한다. 넘버링 bbox를 이미지에 그려서 넣으면 VLM이 그 사각형 자체를 "작은 사각형 하자"로 묘사하는 문제가 실측으로 확인됐다 — 그래서 위치는 텍스트로만 준다(`loaders.build_defect_prompt`가 크롭 방식에서 위치를 안 묻는 것과 같은 원칙, `bbox_grid_position()` 재사용).


In [ ]:
labels_by_inst = [{**inst, "label": qc_shared.KOREAN_LABELS.get(inst["type"], inst["type"])} for inst in kept]

batch_started = time.perf_counter()
batch_raw_response = describe_defects_batch(vlm_bundle, demo_image, labels_by_inst)
batch_elapsed = time.perf_counter() - batch_started
print(f"배치 호출 {batch_elapsed:.1f}초\n원문:\n{batch_raw_response}")

expected_nos = [inst["no"] for inst in kept]
batch_parsed = parse_batch_response(batch_raw_response, expected_nos)
missing = missing_batch_numbers(batch_parsed)
if missing:
    print(f"\n[주의] 번호 {missing}는 배치 응답에서 못 찾았습니다.")
else:
    print("\n모든 번호가 파싱됐습니다.")


## 3. Step 3 — 최종 JSON 조립

`bbox`/`defect_type`/`label`(SAM3) + `description`(VLM, 파싱 실패 시 `None`)만 담는다. **`상태`(사용자 응답)는 여기서 만들지 않는다** — AI는 판정만 하고, 그 finding에 대한 사용자 응답은 BE가 `key`로 관리한다.


In [ ]:
def assemble_finding(inst, description):
    return {
        "no": inst["no"],
        "bbox": [round(v, 4) for v in inst["bbox"]],
        "defect_type": inst["type"],
        "label": qc_shared.KOREAN_LABELS.get(inst["type"], inst["type"]),
        "confidence": round(inst["confidence"], 4),
        "description": description,
    }


final_findings = [assemble_finding(inst, batch_parsed.get(inst["no"])) for inst in kept]
print(jsonlib.dumps(final_findings, ensure_ascii=False, indent=2))


## 4. 이미지 전체 배치 실행 + 결과 저장

§1~§3을 이미지마다 반복해서 최종 JSON을 모은다. 개별 이미지를 다시 자세히 보고 싶으면 `demo_path`/`demo_image`를 바꿔서 §1부터 재실행한다.


In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("./outputs/batch_pipeline_check")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary = []
for path in image_paths:
    image = Image.open(path).convert("RGB")
    row_result = run_one(sam3_caller, image)
    row_kept = row_result["kept"]
    if not row_kept:
        summary.append({"image": path.name, "findings": []})
        continue

    row_insts = [{**inst, "label": qc_shared.KOREAN_LABELS.get(inst["type"], inst["type"])} for inst in row_kept]
    row_response = describe_defects_batch(vlm_bundle, image, row_insts)
    row_expected = [inst["no"] for inst in row_kept]
    row_parsed = parse_batch_response(row_response, row_expected)

    row_findings = [assemble_finding(inst, row_parsed.get(inst["no"])) for inst in row_kept]
    summary.append({"image": path.name, "findings": row_findings})
    missing_count = len(missing_batch_numbers(row_parsed))
    print(f"{path.name}: 하자 {len(row_kept)}건, 누락 {missing_count}건")

out_json = OUTPUT_DIR / "result.json"
out_json.write_text(jsonlib.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n결과 저장 -> {out_json}")


## 결론 (실행 후 채울 것)

- 번호가 누락되거나 헷갈린 사례가 있었는가:
- bbox를 안 보여줬는데도 설명이 그 위치에 맞게 나오는가("사각형" 같은 아티팩트 언급이 사라졌는가):
- 채택할 만한가:

### 후속 작업
- 채택한다면 `docs/plans/AI/`에 `describer.py`/`prompts.py` 반영 설계 계획을 먼저 쓴다(CH006/CH011 선례). 여기 코드를 그대로 승격하지 않는다 — 프로덕션은 계획 문서 승인 후에만 손댄다.
